In [3]:
import pm4py
import pandas as pd

# df = pm4py.read_xes("data/BPI_Challenge_2019.xes")
# df.to_parquet("data/bpi2019.parquet")
# df.shape



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




In [4]:
df = pd.read_parquet("data/bpi2019.parquet")

In [5]:
print(df.columns.tolist())
df.head()

['User', 'org:resource', 'concept:name', 'Cumulative net worth (EUR)', 'time:timestamp', 'case:Spend area text', 'case:Company', 'case:Document Type', 'case:Sub spend area text', 'case:Purchasing Document', 'case:Purch. Doc. Category name', 'case:Vendor', 'case:Item Type', 'case:Item Category', 'case:Spend classification text', 'case:Source', 'case:Name', 'case:GR-Based Inv. Verif.', 'case:Item', 'case:concept:name', 'case:Goods Receipt']


,User,org:resource,concept:name,Cumulative net worth (EUR),time:timestamp,case:Spend area text,case:Company,case:Document Type,case:Sub spend area text,case:Purchasing Document,...,case:Vendor,case:Item Type,case:Item Category,case:Spend classification text,case:Source,case:Name,case:GR-Based Inv. Verif.,case:Item,case:concept:name,case:Goods Receipt
0,batch_00,batch_00,SRM: Created,298.0,2018-01-02 12:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
1,batch_00,batch_00,SRM: Complete,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
2,batch_00,batch_00,SRM: Awaiting Approval,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
3,batch_00,batch_00,SRM: Document Completed,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True
4,batch_00,batch_00,SRM: In Transfer to Execution Syst.,298.0,2018-01-02 13:53:00+00:00,CAPEX & SOCS,companyID_0000,EC Purchase order,Facility Management,2000000000,...,vendorID_0000,Standard,"3-way match, invoice before GR",NPR,sourceSystemID_0000,vendor_0000,False,00001,2000000000_00001,True


In [6]:
df["concept:name"].value_counts()

concept:name
Record Goods Receipt                   314097
Create Purchase Order Item             251734
Record Invoice Receipt                 228760
Vendor creates invoice                 219919
Clear Invoice                          194393
Record Service Entry Sheet             164975
Remove Payment Block                    57136
Create Purchase Requisition Item        46592
Receive Order Confirmation              32065
Change Quantity                         21449
Change Price                            12423
Delete Purchase Order Item               8875
Change Approval for Purchase Order       7541
Cancel Invoice Receipt                   7096
Vendor creates debit memo                6255
Change Delivery Indicator                3289
Cancel Goods Receipt                     3096
SRM: In Transfer to Execution Syst.      1765
SRM: Created                             1628
SRM: Awaiting Approval                   1628
SRM: Document Completed                  1628
SRM: Ordered         

In [7]:
df["case:Item Category"].value_counts()

case:Item Category
3-way match, invoice before GR    1234708
3-way match, invoice after GR      319233
Consignment                         36084
2-way match                          5898
Name: count, dtype: int64

In [8]:
import sqlite3

df = pd.read_parquet("data/bpi2019.parquet")

case_cols = {
    "case:concept:name": "case_id",
    "case:Purchasing Document": "po_number",
    "case:Item": "item_no",
    "case:Vendor": "vendor_id",
    "case:Name": "vendor_name",
    "case:Company": "company_id",
    "case:Document Type": "document_type",
    "case:Item Category": "item_category",
    "case:Item Type": "item_type",
    "case:GR-Based Inv. Verif.": "gr_based_iv",
    "case:Goods Receipt": "goods_receipt",
    "case:Spend area text": "spend_area",
    "case:Sub spend area text": "sub_spend_area",
}

po_items = (df[list(case_cols)]
            .drop_duplicates("case:concept:name")
            .rename(columns=case_cols))
po_items["item_no"] = po_items["item_no"].astype(int)     # "00001" -> 1

vendors = po_items[["vendor_id", "vendor_name"]].drop_duplicates("vendor_id")
po_items = po_items.drop(columns="vendor_name") 

print(len(po_items), "PO items,", len(vendors), "vendors")

251734 PO items, 1975 vendors


In [9]:
events = df[["case:concept:name", "concept:name", "time:timestamp",
             "Cumulative net worth (EUR)", "User"]].rename(columns={
    "case:concept:name": "case_id",
    "concept:name": "activity",
    "time:timestamp": "ts",
    "Cumulative net worth (EUR)": "net_worth",
    "User": "user_id",
})

events = events.sort_values(["case_id", "ts"], kind="stable").reset_index(drop=True)
events.insert(0, "event_id", range(1, len(events) + 1))
events["ts"] = events["ts"].dt.strftime("%Y-%m-%d %H:%M:%S")

events.head()

,event_id,case_id,activity,ts,net_worth,user_id
0,1,2000000000_00001,SRM: Created,2018-01-02 12:53:00,298.0,batch_00
1,2,2000000000_00001,SRM: Complete,2018-01-02 13:53:00,298.0,batch_00
2,3,2000000000_00001,SRM: Awaiting Approval,2018-01-02 13:53:00,298.0,batch_00
3,4,2000000000_00001,SRM: Document Completed,2018-01-02 13:53:00,298.0,batch_00
4,5,2000000000_00001,SRM: In Transfer to Execution Syst.,2018-01-02 13:53:00,298.0,batch_00


In [10]:
con = sqlite3.connect("data/erp.db")

vendors.to_sql("vendors", con, if_exists="replace", index=False)
po_items.to_sql("po_items", con, if_exists="replace", index=False)
events.to_sql("events", con, if_exists="replace", index=False, chunksize=50_000)

con.execute("CREATE INDEX IF NOT EXISTS ix_po ON po_items (po_number, item_no)")
con.execute("CREATE INDEX IF NOT EXISTS ix_events_case ON events (case_id)")
con.execute("CREATE INDEX IF NOT EXISTS ix_events_activity ON events (activity)")
con.commit()

In [11]:
for t in ["vendors", "po_items", "events"]:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

vendors 1975
po_items 251734
events 1595923


In [12]:
# the full history of one line item that has an invoice
sample = con.execute(
    "SELECT case_id FROM events WHERE activity = 'Record Invoice Receipt' LIMIT 1"
).fetchone()[0]

pd.read_sql("SELECT ts, activity, net_worth, user_id FROM events WHERE case_id = ? ORDER BY event_id",
            con, params=[sample])

,ts,activity,net_worth,user_id
0,2018-01-02 12:53:00,SRM: Created,298.0,batch_00
1,2018-01-02 13:53:00,SRM: Complete,298.0,batch_00
2,2018-01-02 13:53:00,SRM: Awaiting Approval,298.0,batch_00
3,2018-01-02 13:53:00,SRM: Document Completed,298.0,batch_00
4,2018-01-02 13:53:00,SRM: In Transfer to Execution Syst.,298.0,batch_00
5,2018-01-02 13:53:00,SRM: Ordered,298.0,batch_00
6,2018-01-02 13:53:00,SRM: Change was Transmitted,298.0,batch_00
7,2018-01-02 13:53:00,Create Purchase Order Item,298.0,user_000
8,2018-01-02 22:59:00,Vendor creates invoice,298.0,NONE
9,2018-03-06 06:44:00,Record Goods Receipt,298.0,user_000


In [13]:
multi = con.execute("""
    SELECT case_id FROM events
    WHERE activity = 'Record Goods Receipt'
    GROUP BY case_id HAVING COUNT(*) >= 3
    LIMIT 1
""").fetchone()[0]

pd.read_sql("SELECT ts, activity, net_worth FROM events WHERE case_id = ? ORDER BY event_id",
            con, params=[multi])

,ts,activity,net_worth
0,2018-01-09 14:30:00,SRM: Created,1668264.0
1,2018-01-09 15:30:00,SRM: Complete,1668264.0
2,2018-01-09 15:30:00,SRM: Awaiting Approval,1668264.0
3,2018-01-09 15:30:00,SRM: Document Completed,1668264.0
4,2018-01-09 15:30:00,SRM: Ordered,1668264.0
5,2018-01-09 15:30:00,SRM: In Transfer to Execution Syst.,1668264.0
6,2018-01-09 15:30:00,SRM: Change was Transmitted,1668264.0
7,2018-01-09 15:30:00,Create Purchase Order Item,1668264.0
8,2018-01-09 22:59:00,Vendor creates invoice,1668264.0
9,2018-01-10 15:11:00,Record Goods Receipt,1668264.0


In [46]:
import json
import ollama
from tools import TOOLS, get_po_item, search_vendors, find_booked_invoices, check_three_way_match

MODEL = "qwen3:8b"

ollama.chat(model=MODEL, messages=[{"role": "user", "content": "Say hi in 3 words."}]).message.content

"Hey, what's up?"

In [17]:
def submit_decision(decision: str, reason_code: str, explanation: str,
                    po_number: str, item_no: int, amount: float, vendor_name: str) -> dict:
    """Submit your final decision for this invoice. Call this exactly once, as your last step.

    Args:
        decision: One of "approve", "reject", "escalate"
        reason_code: One of MATCHED, UNKNOWN_PO, VENDOR_MISMATCH, DUPLICATE, PRICE_VARIANCE,
            MISSING_RECEIPT, HIGH_VALUE, UNCLEAR_INVOICE
        explanation: 2-3 sentences an AP clerk can act on, citing the evidence you found
        po_number: PO number as read from the invoice, digits only
        item_no: Line item number as read from the invoice
        amount: Invoice amount as read from the invoice, as a plain number
        vendor_name: Vendor name exactly as written on the invoice
    """
    return {"status": "recorded"}

AGENT_TOOLS = [get_po_item, search_vendors, find_booked_invoices, check_three_way_match, submit_decision]
TOOL_FUNCS = {**TOOLS, "submit_decision": submit_decision}

In [18]:
REASON_TO_DECISION = {
    "MATCHED": "approve",
    "UNKNOWN_PO": "reject", "VENDOR_MISMATCH": "reject", "DUPLICATE": "reject",
    "PRICE_VARIANCE": "escalate", "MISSING_RECEIPT": "escalate",
    "HIGH_VALUE": "escalate", "UNCLEAR_INVOICE": "escalate",
}

In [19]:
def verify_approval(d, invoice_text):
    """Re-check an approval in code. Returns a problem code, or None if it really is fine."""
    try:
        po, item, amount = str(d["po_number"]), int(d["item_no"]), float(d["amount"])
    except (KeyError, ValueError, TypeError):
        return "UNCLEAR_INVOICE"

    name = d.get("vendor_name", "")
    invoice_digits = "".join(ch for ch in invoice_text if ch.isdigit())
    if not name or name not in invoice_text or po not in invoice_digits:
        return "UNCLEAR_INVOICE"

    m = check_three_way_match(po, item, amount)
    if "error" in m:
        return "UNKNOWN_PO"
    ids = {v["vendor_id"] for v in search_vendors(name) if v["vendor_name"] == name}
    if m["po_vendor_id"] not in ids:
        return "VENDOR_MISMATCH"
    if not m["receipt_ok"]:
        return "MISSING_RECEIPT"
    if not m["amount_within_tolerance"]:
        return "PRICE_VARIANCE"
    if m["above_high_value_threshold"]:
        return "HIGH_VALUE"
    return None

In [45]:
SYSTEM_PROMPT = """You are an accounts-payable agent. You process one vendor invoice at a time and decide
whether to approve it, reject it, or escalate it to a human. You are in a loop: call tools as many
times as you need, then finish by calling submit_decision exactly once.

The invoice is free text, so formats vary. Normalise before calling tools:
- PO numbers are digits only ("4507-001752" -> "4507001752")
- item numbers are integers ("pos. 00002" -> 2)
- amounts are plain numbers ("13.115,18 EUR" -> 13115.18, "EUR 13,115.18" -> 13115.18)

Steps:
1. get_po_item for the PO and line item. If it doesn't exist -> reject / UNKNOWN_PO.
2. find_booked_invoices with the vendor_id, amount, PO number and item number. Only call it a
   DUPLICATE if a result has same_po_line true AND check_three_way_match shows receipt_ok false
   with invoices_already_booked > 0. Invoices on other PO lines are never duplicates.
   A missing receipt with no same-line invoice is MISSING_RECEIPT, not DUPLICATE.
3. check_three_way_match with the amount AND the vendor name exactly as written on the invoice
   (invoice_vendor_name). Never compare or calculate yourself - use its results:
   - vendor_matches_po false -> reject / VENDOR_MISMATCH
   - receipt_ok false -> escalate / MISSING_RECEIPT
   - amount_within_tolerance false -> escalate / PRICE_VARIANCE
   - above_high_value_threshold true -> escalate / HIGH_VALUE
   - everything fine -> approve / MATCHED
4. If you can't read the invoice reliably -> escalate / UNCLEAR_INVOICE.

Base every conclusion on tool results. When unsure, escalate: a wrong approval costs money,
an escalation costs a few minutes."""

In [21]:
def run_agent(invoice_text, max_turns=10, verbose=True):
    messages = [{"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": invoice_text}]

    for turn in range(max_turns):
        response = ollama.chat(model=MODEL, messages=messages, tools=AGENT_TOOLS,
                               options={"temperature": 0}, think=False)
        msg = response.message
        messages.append(msg)

        if not msg.tool_calls:
            if verbose: print("No decision. Model said:", msg.content)
            return {"decision": "escalate", "reason_code": "NO_DECISION", "explanation": msg.content}

        for call in msg.tool_calls:
            name, args = call.function.name, dict(call.function.arguments)
            if name == "submit_decision":
                args["model_decision"] = args.get("decision")          # keep what the model said
                args["decision"] = REASON_TO_DECISION.get(args.get("reason_code"), "escalate")
                if args["decision"] == "approve":
                    problem = verify_approval(args, invoice_text)
                    if problem:
                        args["guardrail"] = f"model approved, but code check found {problem}"
                        args["decision"], args["reason_code"] = "escalate", problem
                if verbose: print(f"\nDECISION: {args}")
                return args
            try:
                result = TOOL_FUNCS[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            if verbose: print(f"-> {name}({args})\n   {str(result)[:300]}")
            messages.append({"role": "tool", "tool_name": name,
                             "content": json.dumps(result, default=str)})

    return {"decision": "escalate", "reason_code": "TURN_LIMIT", "explanation": "Ran out of turns."}

In [38]:
from tools import con, _financials

pick = None
for c in con.execute("""SELECT case_id, po_number, item_no, vendor_id FROM po_items
                        WHERE item_category LIKE '3-way%' ORDER BY RANDOM() LIMIT 1000"""):
    f = _financials(c["case_id"])
    if f["uninvoiced_receipts"] > 0 and 1_000 <= f["line_value"] < 50_000:
        pick = c
        break
print("picked:", dict(pick) if pick else "nothing - run this cell again")

vendor = con.execute("SELECT vendor_name FROM vendors WHERE vendor_id = ?", (pick["vendor_id"],)).fetchone()[0]
po = pick["po_number"]
eu_amount = f"{f['line_value']:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

invoice = f"""INVOICE
From: {vendor}
Invoice no.: VI-20481
Invoice date: 14.03.2019
Your order no.: {po[:4]}-{po[4:]}, pos. {pick['item_no']}

Description: goods/services as ordered
Total due: {eu_amount} EUR
Payment terms: 30 days net."""
print(invoice)

picked: {'case_id': '4508048094_00010', 'po_number': '4508048094', 'item_no': 10, 'vendor_id': 'vendorID_0197'}
INVOICE
From: vendor_0194
Invoice no.: VI-20481
Invoice date: 14.03.2019
Your order no.: 4508-048094, pos. 10

Description: goods/services as ordered
Total due: 4.103,00 EUR
Payment terms: 30 days net.


In [39]:
run_agent(invoice)

-> get_po_item({'item_no': 10, 'po_number': '4508048094'})
   {'po_number': '4508048094', 'item_no': 10, 'vendor_id': 'vendorID_0197', 'vendor_name': 'vendor_0194', 'item_category': '3-way match, invoice before GR', 'goods_receipt_required': True, 'description': 'Trading products (old structure)', 'line_value': 4103.0, 'n_receipts': 1, 'n_invoices': 0, 'uninvo
-> search_vendors({'name': 'vendor_0194'})
   [{'vendor_id': 'vendorID_0197', 'vendor_name': 'vendor_0194'}]
-> find_booked_invoices({'vendor_id': 'vendorID_0197', 'amount': 4103})
   [{'po_number': '4508052331', 'item_no': 10, 'ts': '2018-12-28 14:12:00', 'amount': 4121.0}, {'po_number': '4508049526', 'item_no': 10, 'ts': '2018-12-28 14:08:00', 'amount': 4102.0}, {'po_number': '4508049341', 'item_no': 10, 'ts': '2018-12-28 13:59:00', 'amount': 4119.0}, {'po_number': '4507026047'
-> check_three_way_match({'po_number': '4508048094', 'item_no': 10, 'amount': 4103})
   {'invoice_amount': 4103, 'line_value': 4103.0, 'tolerance': 82.0

{'decision': 'approve',
 'reason_code': 'MATCHED',
 'explanation': 'The invoice matches the purchase order line item, the vendor details align with the PO, and there is a goods receipt recorded. The amount is within the tolerance and no duplicate invoices were found.',
 'po_number': '4508048094',
 'item_no': 10,
 'amount': 4103,
 'vendor_name': 'vendor_0194'}

In [40]:
# price raised 25 % -> expect escalate / PRICE_VARIANCE
run_agent(invoice.replace(eu_amount, f"{f['line_value'] * 1.25:,.2f}"))

-> get_po_item({'po_number': '4508048094', 'item_no': 10})
   {'po_number': '4508048094', 'item_no': 10, 'vendor_id': 'vendorID_0197', 'vendor_name': 'vendor_0194', 'item_category': '3-way match, invoice before GR', 'goods_receipt_required': True, 'description': 'Trading products (old structure)', 'line_value': 4103.0, 'n_receipts': 1, 'n_invoices': 0, 'uninvo
-> search_vendors({'name': 'vendor_0194'})
   [{'vendor_id': 'vendorID_0197', 'vendor_name': 'vendor_0194'}]
-> find_booked_invoices({'vendor_id': 'vendorID_0197', 'amount': 5128.75})
   [{'po_number': '4508054935', 'item_no': 230, 'ts': '2018-12-28 13:42:00', 'amount': 5116.0}, {'po_number': '4508053232', 'item_no': 40, 'ts': '2018-12-10 09:40:00', 'amount': 5112.0}, {'po_number': '4507030342', 'item_no': 70, 'ts': '2018-12-10 09:31:00', 'amount': 5135.0}, {'po_number': '4508042615
-> check_three_way_match({'po_number': '4508048094', 'item_no': 10, 'amount': 5128.75})
   {'invoice_amount': 5128.75, 'line_value': 4103.0, 'toleran

{'decision': 'escalate',
 'reason_code': 'PRICE_VARIANCE',
 'explanation': 'The invoice amount of 5128.75 EUR exceeds the line value of 4103.0 EUR by more than the tolerance of 82.06 EUR. This variance requires further review to ensure accuracy and compliance with purchase order terms.',
 'po_number': '4508048094',
 'item_no': 10,
 'amount': 5128.75,
 'vendor_name': 'vendor_0194'}

In [41]:
# billed by a different vendor -> expect reject / VENDOR_MISMATCH
run_agent(invoice.replace(vendor, "vendor_0001"))

-> get_po_item({'po_number': '4508048094', 'item_no': 10})
   {'po_number': '4508048094', 'item_no': 10, 'vendor_id': 'vendorID_0197', 'vendor_name': 'vendor_0194', 'item_category': '3-way match, invoice before GR', 'goods_receipt_required': True, 'description': 'Trading products (old structure)', 'line_value': 4103.0, 'n_receipts': 1, 'n_invoices': 0, 'uninvo
-> search_vendors({'name': 'vendor_0001'})
   [{'vendor_id': 'vendorID_0001', 'vendor_name': 'vendor_0001'}]
-> find_booked_invoices({'amount': 4103, 'vendor_id': 'vendorID_0001'})
   []
-> check_three_way_match({'po_number': '4508048094', 'item_no': 10, 'amount': 4103})
   {'invoice_amount': 4103, 'line_value': 4103.0, 'tolerance': 82.06, 'amount_within_tolerance': True, 'goods_receipt_required': True, 'receipts': 1, 'invoices_already_booked': 0, 'receipt_ok': True, 'above_high_value_threshold': False, 'po_vendor_id': 'vendorID_0197'}

DECISION: {'explanation': "The invoice matches the purchase order line item, the vendor ID mat

{'explanation': "The invoice matches the purchase order line item, the vendor ID matches the PO's vendor ID, there are no duplicate invoices booked, and the amount is within the tolerance. The goods receipt is confirmed and the invoice is approved.",
 'po_number': '4508048094',
 'item_no': 10,
 'amount': 4103,
 'vendor_name': 'vendor_0001',
 'decision': 'escalate',
 'reason_code': 'VENDOR_MISMATCH',
 'guardrail': 'model approved, but code check found VENDOR_MISMATCH'}

In [22]:
from tools import con, _financials, _find_item, check_three_way_match
import random
from datetime import datetime

con.execute("CREATE TABLE IF NOT EXISTS heldout_events AS SELECT * FROM events WHERE 0")
con.commit()

def rewind(case_id, from_event_id):
    """Hide this event and everything after it for one line item (moves them to heldout_events)."""
    con.execute("INSERT INTO heldout_events SELECT * FROM events WHERE case_id = ? AND event_id >= ?",
                (case_id, from_event_id))
    con.execute("DELETE FROM events WHERE case_id = ? AND event_id >= ?", (case_id, from_event_id))
    con.commit()

def unrewind(case_id):
    """Put one line item's hidden events back."""
    con.execute("INSERT INTO events SELECT * FROM heldout_events WHERE case_id = ?", (case_id,))
    con.execute("DELETE FROM heldout_events WHERE case_id = ?", (case_id,))
    con.commit()

def restore_all():
    """Put every hidden event back, so the database is exactly as it was loaded."""
    con.execute("INSERT INTO events SELECT * FROM heldout_events")
    con.execute("DELETE FROM heldout_events")
    con.commit()

In [27]:
rng = random.Random(42)     # fixed seed, so you get the same test set every time

def eu(x):
    return f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

def render(vendor_name, po, item, amount, inv_no, date):
    amt = rng.choice([f"{eu(amount)} EUR", f"EUR {amount:,.2f}", f"€ {eu(amount)}", f"{amount:.2f} EUR"])
    po_line = rng.choice([f"PO {po} / Item {item:05d}",
                          f"Your order no.: {po[:4]}-{po[4:]}, pos. {item}",
                          f"Purchase Order #{po} (line {item})",
                          f"Ref: {po}_{item:05d}"])
    return f"""{rng.choice(["INVOICE", "Tax Invoice", "Rechnung / Invoice"])}
From: {vendor_name}
Invoice no.: {inv_no}
Invoice date: {date}
{po_line}

Total due: {amt}
{rng.choice(["Payment terms: 30 days net.", "Please pay within 14 days.", "Thank you for your business."])}"""

In [29]:
raise SystemExit("Don't re-run: this rebuilds the test set.")

restore_all()          # start from the full database every time you run this cell

targets = {"clean": 14, "high_value": 4, "price_inflated": 6,
           "missing_receipt": 6, "vendor_mismatch": 5, "unknown_po": 5}
counts = {k: 0 for k in targets}
tests = []
all_vendors = [r["vendor_name"] for r in con.execute("SELECT vendor_name FROM vendors")]

cases = con.execute("""
    SELECT p.case_id, p.po_number, p.item_no, v.vendor_name
    FROM po_items p JOIN vendors v USING (vendor_id)
    WHERE p.item_category LIKE '3-way%'
      AND p.case_id IN (SELECT case_id FROM events WHERE activity = 'Record Invoice Receipt')
    ORDER BY RANDOM() LIMIT 5000""").fetchall()

def add(c, scenario, expected, vendor_name, po, amount, ir_ts):
    date = datetime.strptime(ir_ts[:10], "%Y-%m-%d").strftime("%d.%m.%Y")
    tests.append({
        "test_id": f"T{len(tests) + 1:03d}", "case_id": c["case_id"], "scenario": scenario,
        "expected_decision": expected[0], "expected_reason": expected[1],
        "invoice_text": render(vendor_name, po, c["item_no"], amount, f"VI-{rng.randint(10000, 99999)}", date),
    })
    counts[scenario] += 1

for c in cases:
    if all(counts[k] >= targets[k] for k in targets):
        break
    irs = con.execute("""SELECT event_id, ts FROM events
                         WHERE case_id = ? AND activity = 'Record Invoice Receipt'
                         ORDER BY event_id""", (c["case_id"],)).fetchall()
    first_ir, last_ir = irs[0], irs[-1]

    # Missing receipt: rewind to the FIRST invoice and check whether nothing had been received yet
    if counts["missing_receipt"] < targets["missing_receipt"]:
        rewind(c["case_id"], first_ir["event_id"])
        f = _financials(c["case_id"])
        m = check_three_way_match(c["po_number"], c["item_no"], f["line_value"])
        if 1_000 <= f["line_value"] < 50_000 and m["goods_receipt_required"] and not m["receipt_ok"]:
            add(c, "missing_receipt", ("escalate", "MISSING_RECEIPT"),
                c["vendor_name"], c["po_number"], f["line_value"], first_ir["ts"])
            continue
        unrewind(c["case_id"])

    # Everything else: rewind to the LAST invoice, which should be a normal, payable invoice
    rewind(c["case_id"], last_ir["event_id"])
    f = _financials(c["case_id"])
    m = check_three_way_match(c["po_number"], c["item_no"], f["line_value"])
    if not (m["receipt_ok"] and f["line_value"] >= 1_000):
        unrewind(c["case_id"])
        continue

    line, vendor, po = f["line_value"], c["vendor_name"], c["po_number"]
    if m["above_high_value_threshold"]:
        if counts["high_value"] < targets["high_value"]:
            add(c, "high_value", ("escalate", "HIGH_VALUE"), vendor, po, line, last_ir["ts"])
        else:
            unrewind(c["case_id"])
        continue

    scenario = next((s for s in ["price_inflated", "vendor_mismatch", "unknown_po", "clean"]
                     if counts[s] < targets[s]), None)
    if scenario is None:
        unrewind(c["case_id"])
    elif scenario == "price_inflated":
        add(c, scenario, ("escalate", "PRICE_VARIANCE"), vendor, po,
            round(line * rng.uniform(1.10, 1.40), 2), last_ir["ts"])
    elif scenario == "vendor_mismatch":
        other = rng.choice([v for v in all_vendors if v != vendor])
        add(c, scenario, ("reject", "VENDOR_MISMATCH"), other, po, line, last_ir["ts"])
    elif scenario == "unknown_po":
        fake = po
        while _find_item(fake, c["item_no"]) is not None:
            fake = po[:-3] + f"{rng.randint(0, 999):03d}"
        add(c, scenario, ("reject", "UNKNOWN_PO"), vendor, fake, line, last_ir["ts"])
    else:
        add(c, scenario, ("approve", "MATCHED"), vendor, po, line, last_ir["ts"])

test_df = pd.DataFrame(tests)
test_df.to_sql("test_invoices", con, if_exists="replace", index=False)
print(counts)

SystemExit: Don't re-run: this rebuilds the test set.

C:\Users\jdev4\AppData\Roaming\Python\Python313\site-packages\IPython\core\interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [31]:
test_df = pd.read_sql("SELECT * FROM test_invoices", con)

In [32]:
for s, row in test_df.groupby("scenario").head(1).set_index("scenario").iterrows():
    print(f"--- {s}  (expected: {row.expected_decision} / {row.expected_reason})")
    print(row.invoice_text, "\n")

--- price_inflated  (expected: escalate / PRICE_VARIANCE)
INVOICE
From: vendor_0648
Invoice no.: VI-13278
Invoice date: 23.06.2018
Your order no.: 4507-023085, pos. 190

Total due: € 13.418,22
Payment terms: 30 days net. 

--- high_value  (expected: escalate / HIGH_VALUE)
Tax Invoice
From: vendor_0181
Invoice no.: VI-81426
Invoice date: 29.11.2018
Your order no.: 4507-040888, pos. 10

Total due: 55744.00 EUR
Thank you for your business. 

--- vendor_mismatch  (expected: reject / VENDOR_MISMATCH)
Rechnung / Invoice
From: vendor_0873
Invoice no.: VI-26361
Invoice date: 11.12.2018
PO 4508063153 / Item 00030

Total due: 3491.00 EUR
Please pay within 14 days. 

--- unknown_po  (expected: reject / UNKNOWN_PO)
Rechnung / Invoice
From: vendor_0106
Invoice no.: VI-44993
Invoice date: 14.09.2018
Your order no.: 4508-044686, pos. 80

Total due: 3.105,00 EUR
Thank you for your business. 

--- missing_receipt  (expected: escalate / MISSING_RECEIPT)
INVOICE
From: vendor_0151
Invoice no.: VI-76784
In

In [47]:
import time

results = []
for i, t in enumerate(test_df.itertuples(), 1):
    start = time.time()
    try:
        out = run_agent(t.invoice_text, verbose=False)
    except Exception as e:                       # one crash shouldn't stop the whole run
        out = {"decision": "error", "reason_code": type(e).__name__, "explanation": str(e)}
    secs = round(time.time() - start, 1)
    results.append({
        "test_id": t.test_id, "scenario": t.scenario,
        "expected_decision": t.expected_decision, "expected_reason": t.expected_reason,
        "decision": out.get("decision"), "reason_code": out.get("reason_code"),
        "guardrail": out.get("guardrail"), "explanation": out.get("explanation"), "seconds": secs,
        "model_decision": out.get("model_decision"),
    })
    mark = "OK " if out.get("decision") == t.expected_decision else "XX "
    print(f"{mark}{i:>2}/{len(test_df)}  {t.scenario:<16} expected {t.expected_decision:<9}"
          f"got {str(out.get('decision')):<9} ({secs}s)")

res = pd.DataFrame(results)
res.to_csv("results_qwen3_8b.csv", index=False)

OK  1/40  price_inflated   expected escalate got escalate  (8.7s)
OK  2/40  price_inflated   expected escalate got escalate  (7.2s)
OK  3/40  price_inflated   expected escalate got escalate  (7.8s)
OK  4/40  high_value       expected escalate got escalate  (8.2s)
OK  5/40  price_inflated   expected escalate got escalate  (8.1s)
OK  6/40  price_inflated   expected escalate got escalate  (8.4s)
OK  7/40  price_inflated   expected escalate got escalate  (8.2s)
OK  8/40  vendor_mismatch  expected reject   got reject    (7.5s)
OK  9/40  vendor_mismatch  expected reject   got reject    (8.1s)
OK 10/40  vendor_mismatch  expected reject   got reject    (8.5s)
XX 11/40  vendor_mismatch  expected reject   got escalate  (7.5s)
OK 12/40  vendor_mismatch  expected reject   got reject    (7.5s)
OK 13/40  unknown_po       expected reject   got reject    (4.0s)
OK 14/40  unknown_po       expected reject   got reject    (8.1s)
OK 15/40  unknown_po       expected reject   got reject    (6.8s)
OK 16/40  

In [48]:
res["correct"] = res.decision == res.expected_decision
res["model_decision"] = res.decision.where(res.guardrail.isna(), "approve")   # what the model said before the guardrail

problems = res[res.expected_decision != "approve"]
clean = res[res.expected_decision == "approve"]

pd.Series({
    "Decision accuracy (final)":           f"{res.correct.mean():.0%}",
    "Decision accuracy (model alone)":     f"{(res.model_decision == res.expected_decision).mean():.0%}",
    "Problem invoices wrongly approved":   f"{(problems.decision == 'approve').sum()} of {len(problems)}",
    "  ...by the model, before guardrail": f"{(problems.model_decision == 'approve').sum()} of {len(problems)}",
    "Clean invoices wrongly flagged":      f"{(clean.decision != 'approve').sum()} of {len(clean)}",
    "Guardrail overrides":                 int(res.guardrail.notna().sum()),
    "Straight-through (no human needed)":  f"{res.decision.isin(['approve', 'reject']).mean():.0%}",
    "Avg seconds per invoice":             round(res.seconds.mean(), 1),
})

Decision accuracy (final)                  98%
Decision accuracy (model alone)            92%
Problem invoices wrongly approved      0 of 26
  ...by the model, before guardrail    3 of 26
Clean invoices wrongly flagged         0 of 14
Guardrail overrides                          3
Straight-through (no human needed)         57%
Avg seconds per invoice                    7.8
dtype: object

In [49]:
res.groupby("scenario").correct.agg(correct="sum", total="count")
pd.crosstab(res.expected_decision, res.decision)      # rows = right answer, columns = what it decided
res[~res.correct][["test_id", "scenario", "expected_reason", "decision", "reason_code", "guardrail", "explanation"]]

,test_id,scenario,expected_reason,decision,reason_code,guardrail,explanation
10,T011,vendor_mismatch,VENDOR_MISMATCH,escalate,VENDOR_MISMATCH,"model approved, but code check found VENDOR_MI...",The invoice matches the purchase order line it...
